In [ ]:
# Data
import pandas as pd
import numpy as np

# Visualization
import matplotlib.pyplot as plt
import seaborn as sns

# Preprocessing
from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import train_test_split

# ML models
from sklearn.ensemble import RandomForestRegressor
from sklearn.ensemble import RandomForestClassifier
from sklearn.ensemble import IsolationForest
from sklearn.svm import OneClassSVM

# Boosting models
from xgboost import XGBClassifier
from lightgbm import LGBMClassifier

# Evaluation
from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    confusion_matrix,
    classification_report
)

# Utilities
import joblib

In [ ]:
data = pd.read_parquet("../data/features/final_features_with_context.parquet")

print(data.shape)
data.head()

In [ ]:
data.info()

data.describe().T

In [ ]:
data.isnull().sum().sort_values(ascending=False)

In [ ]:
from sklearn.preprocessing import LabelEncoder

le = LabelEncoder()

data["context_encoded"] = le.fit_transform(data["context_label"])

In [ ]:
stat_features = [
'speed_mean','speed_std','speed_max','speed_min',
'rpm_mean','rpm_std','rpm_max','rpm_min',
'throttle_mean','throttle_std',
'map_mean','map_std',
'maf_mean','maf_std',
'acceleration_mean','acceleration_std',
'jerk_mean','jerk_std'
]

event_features = [
'harsh_acc_events',
'harsh_brake_events',
'engine_stress_events',
'high_map_events'
]

behavior_features = [
'aggression_score',
'control_instability',
'speed_stability',
'acc_variability',
'throttle_smoothness',
'acc_smoothness',
'jerk_intensity'
]

mechanical_features = [
'engine_load',
'load_indicator',
'rpm_speed_ratio',
'rpm_speed_ratio_norm',
'cold_start'
]

context_features = [
'context_cluster',
'context_encoded'
]

In [ ]:
feature_columns = (
    stat_features +
    event_features +
    behavior_features +
    mechanical_features +
    context_features
)

print(len(feature_columns))

In [ ]:
from sklearn.preprocessing import StandardScaler

scaler = StandardScaler()

data[feature_columns] = scaler.fit_transform(data[feature_columns])

In [ ]:
trip_ids = sorted(data.trip_id.unique())

split_index = int(len(trip_ids) * 0.8)

train_ids = trip_ids[:split_index]
test_ids = trip_ids[split_index:]

train = data[data.trip_id.isin(train_ids)].copy()
test = data[data.trip_id.isin(test_ids)].copy()

print("Train trips:", len(train_ids))
print("Test trips:", len(test_ids))
print("Train rows:", train.shape)
print("Test rows:", test.shape)

In [ ]:
threshold = train["aggression_score"].quantile(0.80)

train["behavior_label"] = (train["aggression_score"] > threshold).astype(int)
test["behavior_label"] = (test["aggression_score"] > threshold).astype(int)

print("Aggressive threshold:", threshold)

In [ ]:
from sklearn.preprocessing import StandardScaler

scaler = StandardScaler()

train.loc[:, feature_columns] = scaler.fit_transform(train[feature_columns])
test.loc[:, feature_columns] = scaler.transform(test[feature_columns])

In [ ]:
from sklearn.ensemble import RandomForestClassifier

rf_model = RandomForestClassifier(
    n_estimators=500,
    max_depth=18,
    min_samples_split=5,
    min_samples_leaf=2,
    max_features="sqrt",
    bootstrap=True,
    n_jobs=-1,
    random_state=42
)

rf_model.fit(train[feature_columns], train["behavior_label"])

rf_pred = rf_model.predict(test[feature_columns])

In [ ]:
from xgboost import XGBClassifier

xgb_model = XGBClassifier(
    n_estimators=500,
    learning_rate=0.03,
    max_depth=8,
    subsample=0.8,
    colsample_bytree=0.8,
    reg_alpha=0.5,
    reg_lambda=1,
    objective="binary:logistic",
    eval_metric="logloss",
    random_state=42,
    n_jobs=-1
)

xgb_model.fit(train[feature_columns], train["behavior_label"])

xgb_pred = xgb_model.predict(test[feature_columns])

In [ ]:
from lightgbm import LGBMClassifier

lgb_model = LGBMClassifier(
    n_estimators=600,
    learning_rate=0.03,
    max_depth=-1,
    num_leaves=64,
    subsample=0.8,
    colsample_bytree=0.8,
    class_weight="balanced",
    random_state=42,
    n_jobs=-1
)

lgb_model.fit(train[feature_columns], train["behavior_label"])

lgb_pred = lgb_model.predict(test[feature_columns])

In [ ]:
from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    confusion_matrix,
    classification_report
)

In [ ]:
def evaluate_model(name, y_true, y_pred):

    print(f"\n{name}")
    print("-"*40)

    print("Accuracy:", accuracy_score(y_true, y_pred))
    print("Precision:", precision_score(y_true, y_pred))
    print("Recall:", recall_score(y_true, y_pred))
    print("F1 Score:", f1_score(y_true, y_pred))

    print("\nConfusion Matrix")
    print(confusion_matrix(y_true, y_pred))

    print("\nClassification Report")
    print(classification_report(y_true, y_pred))

In [ ]:
evaluate_model("Random Forest", test["behavior_label"], rf_pred)

evaluate_model("XGBoost", test["behavior_label"], xgb_pred)

evaluate_model("LightGBM", test["behavior_label"], lgb_pred)

In [ ]:
from sklearn.ensemble import StackingClassifier
from xgboost import XGBClassifier

In [ ]:
stack_model = StackingClassifier(

    estimators=[
        ("rf", rf_model),
        ("xgb", xgb_model),
        ("lgb", lgb_model)
    ],

    final_estimator=XGBClassifier(
        n_estimators=200,
        learning_rate=0.05,
        max_depth=4,
        subsample=0.8,
        colsample_bytree=0.8,
        random_state=42,
        n_jobs=-1
    ),

    passthrough=False,
    n_jobs=-1
)

In [ ]:
stack_model.fit(train[feature_columns], train["behavior_label"])

In [ ]:
stack_pred = stack_model.predict(test[feature_columns])

In [ ]:
evaluate_model("STACKING MODEL", test["behavior_label"], stack_pred)

In [ ]:
stack_prob = stack_model.predict_proba(test[feature_columns])[:,1]

In [ ]:
test = test.copy()
test["aggressive_probability"] = stack_prob

# Also predict for ALL data
all_prob = stack_model.predict_proba(data[feature_columns])[:,1]

data = data.copy()
data["aggressive_probability"] = all_prob

In [ ]:
# risk score for test (for evaluation)
test["risk_score"] = np.clip(
    np.log1p(test["aggressive_probability"] * 9) * 50,
    0,
    100
)

# risk score for ALL trips
data["risk_score"] = np.clip(
    np.log1p(data["aggressive_probability"] * 9) * 50,
    0,
    100
)

In [ ]:
def risk_category(score):

    if score < 20:
        return "Safe"

    elif score < 40:
        return "Low Risk"

    elif score < 60:
        return "Moderate Risk"

    elif score < 80:
        return "High Risk"

    else:
        return "Dangerous"

In [ ]:
test["risk_category"] = test["risk_score"].apply(risk_category)
data["risk_category"] = data["risk_score"].apply(risk_category)

In [ ]:
test[[
    "trip_id",
    "risk_score",
    "risk_category",
    "aggressive_probability"
]].head()

In [ ]:
trip_risk = data.groupby("trip_id").agg({
    "risk_score": "mean",
    "aggressive_probability": "mean"
}).reset_index()

In [ ]:
trip_risk["trip_risk_category"] = trip_risk["risk_score"].apply(risk_category)

In [ ]:
trip_risk.head()

In [ ]:
import seaborn as sns
import matplotlib.pyplot as plt

plt.figure(figsize=(8,5))

sns.histplot(data["risk_score"], bins=30)

plt.title("Driver Risk Score Distribution")

plt.xlabel("Risk Score")

plt.show()

In [ ]:
plt.figure(figsize=(12,6))

sns.barplot(
    x="trip_id",
    y="risk_score",
    data=trip_risk
)

plt.xticks(rotation=90)

plt.title("Trip Risk Scores")

plt.show()

In [ ]:
feature_importance = pd.Series(
    rf_model.feature_importances_,
    index=feature_columns
).sort_values(ascending=False)

In [ ]:
feature_importance.head(15)

In [ ]:
plt.figure(figsize=(8,6))

feature_importance.head(15).plot(kind="barh")

plt.title("Top Driving Behaviour Features")

plt.gca().invert_yaxis()

plt.show()

In [ ]:
data.to_csv("../reports/driver_risk_predictions.csv", index=False)

In [ ]:
trip_risk.to_csv("../reports/trip_risk_summary.csv", index=False)

In [ ]:
import joblib

joblib.dump(stack_model, "../models/driver_behavior_model.pkl")

In [ ]:
print("Driver Behaviour Analysis System - Version 1 Completed")